# 03a_batch_report: Reproducible batch pipeline

Goal: convert the lazy pipeline into a CLI-style script with configuration, logging, and quick validation artifacts.

## Files

```
lectures/02/demo/
  03a_batch_report.md (this walkthrough)
  03a_batch_report.py  # optional script version
  data/...             # from prior demos
  outputs/
```

## Steps

1. **Create a config file** (`config/polars_report.yaml`)
    ```yaml
    source_glob: "lectures/02/demo/data/admissions/part=*/admissions_*.parquet"
    labs_glob: "lectures/02/demo/data/labs/part=*/labs_*.parquet"
    start_date: "2023-01-01"
    summary_path: "lectures/02/demo/outputs/monthly_summary.parquet"
    dashboard_csv: "lectures/02/demo/outputs/monthly_summary.csv"
    chart_path: "lectures/02/demo/outputs/monthly_summary.png"
    ```

2. **Write the batch script**
    ```python
    #%% 03a_batch_report.py
    import argparse
    import logging
    from pathlib import Path
    import polars as pl

    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="config/polars_report.yaml")
    args = parser.parse_args()

    import yaml
    cfg = yaml.safe_load(Path(args.config).read_text())

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

    admissions = pl.scan_parquet(cfg["source_glob"]).select([
        "admission_id", "patient_id", "facility", "admit_ts"
    ])

    labs = pl.scan_parquet(cfg["labs_glob"]).filter(
        pl.col("lab_ts") >= pl.datetime(cfg["start_date"])
    )

    query = (
        labs.join(admissions, on="admission_id", how="inner")
        .group_by([
            "facility",
            pl.col("lab_ts").dt.year().alias("year"),
            pl.col("lab_ts").dt.month().alias("month")
        ])
        .agg([
            pl.count().alias("num_tests"),
            pl.mean("result_value").alias("avg_hba1c"),
            pl.std("result_value").alias("std_hba1c")
        ])
        .sort(["facility", "year", "month"])
    )

    summary = query.collect(streaming=True)
    Path(cfg["summary_path"]).parent.mkdir(parents=True, exist_ok=True)
    summary.write_parquet(cfg["summary_path"])
    summary.write_csv(cfg["dashboard_csv"])

    logging.info("Wrote %s rows", summary.height)
    ```

3. **Add a quick validation notebook/script**
    ```python
    #%% 03b_validate_output.py
    import polars as pl
    df = pl.read_parquet("lectures/02/demo/outputs/monthly_summary.parquet")
    assert df.height > 0
    assert df["avg_hba1c"].is_finite().all()
    print(df.head())
    ```

4. **Optional: create a lightweight plot**
    ```python
    import altair as alt

    chart = alt.Chart(summary.to_pandas()).mark_line().encode(
        x="month:O",
        y="avg_hba1c:Q",
        color="facility:N"
    )
    chart.save(cfg["chart_path"])
    ```

## Checkpoints

- Running `uv run python 03a_batch_report.py --config config/polars_report.yaml` creates Parquet + CSV + optional PNG
- Validation script confirms schema/row counts
- Logs show start/end with row counts and duration